# GP - SWIM Experiments

In [1]:
import torch
import gpytorch
import numpy as np
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, '/Users/gizemnurdal/Workspace/swim-meets-kans')

import importlib
from sgkan.gaussian_process_models import (
    init_gp,
    train_gp,
    predict
)
from sgkan import evaluation_metrics as em


## STAGE 1: Create a TOY dataset and fit an Exact Gaussian Process 

In [2]:
torch.manual_seed(42)

# ─── 1. Create dataset ───────────────────────────────────
N_train = 100
N_test  = 300

# Input: uniform in [-3, 3]
X_train = torch.linspace(-3, 3, N_train).unsqueeze(1)  # shape (100, 1)
y_train = torch.sin(X_train.squeeze()) + 0.1 * torch.randn(N_train)

X_test  = torch.linspace(-4, 4, N_test).unsqueeze(1)   # shape (300, 1)
y_test  = torch.sin(X_test.squeeze())                   # noiseless ground truth

print(f"X_train: {X_train.shape},  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape},   y_test:  {y_test.shape}")


X_train: torch.Size([100, 1]),  y_train: torch.Size([100])
X_test:  torch.Size([300, 1]),   y_test:  torch.Size([300])


In [3]:
# ─── 2. Define GP model ──────────────────────────────────
class ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, X_train, y_train, likelihood):
        super().__init__(X_train, y_train, likelihood)
        self.mean_module  = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.RBFKernel(ard_num_dims=X_train.shape[1])
        )

    def forward(self, x):
        mean  = self.mean_module(x)
        covar = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean, covar) # type: ignore

In [4]:
# ─── 3. Initialize ───────────────────────────────────────
likelihood = gpytorch.likelihoods.GaussianLikelihood()
model      = ExactGPModel(X_train, y_train, likelihood)

In [5]:
# ─── 4. Train ────────────────────────────────────────────
model.train()
likelihood.train()

optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
mll       = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

num_iters = 100 
for i in range(num_iters):
    optimizer.zero_grad()
    loss = -mll(model(X_train), y_train) # type: ignore
    loss.backward()
    optimizer.step()

print(f"\nGP fitted successfully.")
print(f"  Length scale: {model.covar_module.base_kernel.lengthscale.item():.4f}")
print(f"  Output scale: {model.covar_module.outputscale.item():.4f}")
print(f"  Noise:        {likelihood.noise.item():.4f}")
print(f"  Mean const:   {model.mean_module.constant.item():.4f}") # type: ignore


GP fitted successfully.
  Length scale: 1.2596
  Output scale: 0.6832
  Noise:        0.0089
  Mean const:   0.1524


In [6]:
# ─── 5. Freeze GP ────────────────────────────────────────
model.eval()
likelihood.eval()
print(f"\nGP frozen. Ready for Stage 2 — pair sampling.")


GP frozen. Ready for Stage 2 — pair sampling.


## STAGE 2: GP Driven SWIM Scores

In [7]:
"""
    X_train,          # torch tensor (N, d)
    y_train,          # torch tensor (N,)
    model,            # fitted frozen GP model
    likelihood,       # fitted frozen likelihood
    M,                # number of candidate pairs
    N_pairs,          # number of pairs to select (= layer_width equivalent)
    T=3,              # number of interior points per pair
    epsilon=1e-8,     # numerical stability
    random_seed=42
"""

random_seed = 42
rng = np.random.default_rng(random_seed)
N = X_train.shape[0]

In [8]:
# ── Step 1: Sample M candidate pairs ─────────────────
# Same logic as SWIM — delta trick guarantees idx_from != idx_to
M = 100 # Update later
idx_from = rng.integers(low=0, high=N, size=M)
delta    = rng.integers(low=1, high=N-1, size=M)
idx_to   = (idx_from + delta) % N

# Select corr. values using the indices list
x_a = X_train[idx_from]   # shape (M, d)
x_b = X_train[idx_to]     # shape (M, d)
y_a = y_train[idx_from]   # shape (M,)
y_b = y_train[idx_to]   # shape (M,)

In [9]:
# ── Step 2: Create T interior points per pair ────────
# t in {1/(T+1), 2/(T+1), ..., T/(T+1)} — avoids endpoints
T = 3
t_values = torch.linspace(0, 1, T+2)[1:-1]  # shape (T,)
# x_t shape: (M, T, d)
# x_a[:, None, :] broadcasts to (M, 1, d)
x_interior = (
    x_a.unsqueeze(1) +
    t_values.view(1, T, 1) * (x_b - x_a).unsqueeze(1)
)  # (M, T, d)
# Flatten to (M*T, d) for single GP query
x_interior_flat = x_interior.reshape(M * T, -1)

In [10]:
# ── Step 3: Query frozen GP at interior points ───────
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    pred         = likelihood(model(x_interior_flat))
    mu_interior  = pred.mean.reshape(M, T)      # (M, T)
    std_interior = pred.variance.sqrt().reshape(M, T)  # (M, T)

# ── Endpoint gradients: need grad through mu ──
x_a_g = x_a.detach().requires_grad_(True)  # (M, d)
x_b_g = x_b.detach().requires_grad_(True)  # (M, d)

with gpytorch.settings.fast_pred_var():
    pred_a = likelihood(model(x_a_g))
    pred_b = likelihood(model(x_b_g))
    
    mu_a  = pred_a.mean          # (M,)
    std_a = pred_a.variance.sqrt()  # (M,)
    
    mu_b  = pred_b.mean          # (M,)
    std_b = pred_b.variance.sqrt()  # (M,)

# ── Numerator: L-inf norm of gradient difference ──
grad_a = torch.autograd.grad(mu_a.sum(), x_a_g)[0]  # (M, d)
grad_b = torch.autograd.grad(mu_b.sum(), x_b_g)[0]  # (M, d)

numerator = (grad_a - grad_b).abs().max(dim=1).values  # (M,)

# ── Denominator: uncertainty at endpoints + along segment ──
epsilon = 1e-6
denominator = std_a + std_interior.sum(dim=1) + std_b + epsilon  # (M,) # dont add boundaries

# ── Scores and probabilities ──
scores      = numerator / denominator          # (M,)
probs       = scores / scores.sum()            # (M,)  sums to 1

In [11]:
scores

tensor([0.3092, 1.3311, 0.7313, 0.0910, 1.8451, 2.5379, 0.6513, 0.3774, 1.1018,
        1.0275, 2.9453, 1.6697, 0.3155, 0.1443, 0.4672, 1.3777, 0.0789, 1.4221,
        0.8127, 0.0817, 3.8840, 3.6086, 0.5392, 2.9557, 2.0383, 1.5123, 0.6715,
        2.9231, 3.5342, 3.7996, 4.5547, 0.3372, 1.1981, 3.6418, 2.3502, 3.1354,
        3.0878, 0.3220, 1.2265, 3.4800, 0.3890, 1.5525, 2.1770, 1.2584, 0.2641,
        0.9387, 0.2796, 0.2247, 1.2866, 0.5704, 2.0429, 2.4280, 0.8304, 2.2096,
        3.7722, 2.9257, 1.3369, 1.2341, 0.1498, 1.2296, 0.1069, 1.5403, 1.4667,
        2.7806, 2.4701, 1.8794, 1.4103, 1.0354, 3.4078, 0.1628, 0.8658, 0.7386,
        2.0954, 0.2177, 0.4402, 3.7205, 3.1107, 1.3345, 0.1499, 1.6617, 0.4568,
        0.1760, 1.7988, 3.2124, 2.8657, 1.3641, 1.8656, 0.0196, 2.3997, 0.0760,
        3.1133, 2.3889, 2.9255, 1.3306, 0.3590, 0.7886, 2.0231, 1.4303, 3.1236,
        0.7334], grad_fn=<DivBackward0>)

In [12]:
probs

tensor([0.0020, 0.0084, 0.0046, 0.0006, 0.0117, 0.0160, 0.0041, 0.0024, 0.0070,
        0.0065, 0.0186, 0.0106, 0.0020, 0.0009, 0.0030, 0.0087, 0.0005, 0.0090,
        0.0051, 0.0005, 0.0245, 0.0228, 0.0034, 0.0187, 0.0129, 0.0096, 0.0042,
        0.0185, 0.0223, 0.0240, 0.0288, 0.0021, 0.0076, 0.0230, 0.0148, 0.0198,
        0.0195, 0.0020, 0.0077, 0.0220, 0.0025, 0.0098, 0.0138, 0.0080, 0.0017,
        0.0059, 0.0018, 0.0014, 0.0081, 0.0036, 0.0129, 0.0153, 0.0052, 0.0140,
        0.0238, 0.0185, 0.0084, 0.0078, 0.0009, 0.0078, 0.0007, 0.0097, 0.0093,
        0.0176, 0.0156, 0.0119, 0.0089, 0.0065, 0.0215, 0.0010, 0.0055, 0.0047,
        0.0132, 0.0014, 0.0028, 0.0235, 0.0197, 0.0084, 0.0009, 0.0105, 0.0029,
        0.0011, 0.0114, 0.0203, 0.0181, 0.0086, 0.0118, 0.0001, 0.0152, 0.0005,
        0.0197, 0.0151, 0.0185, 0.0084, 0.0023, 0.0050, 0.0128, 0.0090, 0.0197,
        0.0046], grad_fn=<DivBackward0>)

In [13]:
probs.sum()

tensor(1.0000, grad_fn=<SumBackward0>)

In [14]:
# ── Step 5: Sample winning pairs ──
layer_width = 10
probs_np = probs.detach().cpu().numpy()  # multinomial needs numpy for rng.choice

selected_idx = rng.choice(
    M,                        # sample from M candidates
    size=layer_width,         # pick layer_width winners
    replace=True,             # same pair can be selected multiple times
    p=probs_np
)

# Index into your pair tensors
x_a_selected = x_a[selected_idx]  # (layer_width, d)
x_b_selected = x_b[selected_idx]  # (layer_width, d)

In [15]:
x_a_selected

tensor([[ 1.8485],
        [-2.5758],
        [-0.3333],
        [ 1.1212],
        [-1.6667],
        [-0.2121],
        [ 1.2424],
        [-0.3333],
        [ 1.3030],
        [-0.3939]])

In [16]:
x_b_selected

tensor([[-0.4545],
        [-0.5758],
        [-2.5758],
        [ 1.8485],
        [-0.6970],
        [-1.6667],
        [ 2.9394],
        [-2.5758],
        [-1.7273],
        [-2.3333]])

In [17]:
# ── Step 7: Sample GP posterior functions over selected segments ──

# Create dense interior points for each selected pair (for smooth function)
T_sample = 200  # more points for a smooth curve equivalent to 50
t_dense  = torch.linspace(0, 1, T_sample)  # (T_sample,)

# Interior points for selected pairs only
x_segments = (
    x_a_selected.unsqueeze(1) +
    t_dense.view(1, T_sample, 1) * (x_b_selected - x_a_selected).unsqueeze(1)
)  # (layer_width, T_sample, d)

# Flatten for GP query
x_segments_flat = x_segments.reshape(layer_width * T_sample, -1)  # (layer_width*T_sample, d)

# Get posterior distribution over these points
with gpytorch.settings.fast_pred_var():
    pred_segments = likelihood(model(x_segments_flat))

# Reshape mean and covariance for sampling
# We need to sample per segment separately
sampled_functions = []

for i in range(layer_width):
    # Points for this segment
    x_seg_i = x_segments[i]  # (T_sample, d)
    
    with gpytorch.settings.fast_pred_var():
        pred_i = likelihood(model(x_seg_i))
    
    # Sample one function from the posterior
    f_sample = pred_i.mean  # (T_sample,)
    sampled_functions.append(f_sample)

sampled_functions = torch.stack(sampled_functions)  # (layer_width, T_sample)

In [18]:
x_segments.shape, x_segments[0]

(torch.Size([10, 200, 1]),
 tensor([[ 1.8485],
         [ 1.8369],
         [ 1.8253],
         [ 1.8138],
         [ 1.8022],
         [ 1.7906],
         [ 1.7790],
         [ 1.7675],
         [ 1.7559],
         [ 1.7443],
         [ 1.7328],
         [ 1.7212],
         [ 1.7096],
         [ 1.6980],
         [ 1.6865],
         [ 1.6749],
         [ 1.6633],
         [ 1.6517],
         [ 1.6402],
         [ 1.6286],
         [ 1.6170],
         [ 1.6055],
         [ 1.5939],
         [ 1.5823],
         [ 1.5707],
         [ 1.5592],
         [ 1.5476],
         [ 1.5360],
         [ 1.5244],
         [ 1.5129],
         [ 1.5013],
         [ 1.4897],
         [ 1.4781],
         [ 1.4666],
         [ 1.4550],
         [ 1.4434],
         [ 1.4319],
         [ 1.4203],
         [ 1.4087],
         [ 1.3971],
         [ 1.3856],
         [ 1.3740],
         [ 1.3624],
         [ 1.3508],
         [ 1.3393],
         [ 1.3277],
         [ 1.3161],
         [ 1.3046],
         [ 1.

In [19]:
sampled_functions[0]

tensor([ 9.4214e-01,  9.4585e-01,  9.4946e-01,  9.5296e-01,  9.5635e-01,
         9.5964e-01,  9.6283e-01,  9.6590e-01,  9.6887e-01,  9.7173e-01,
         9.7448e-01,  9.7712e-01,  9.7964e-01,  9.8206e-01,  9.8436e-01,
         9.8655e-01,  9.8863e-01,  9.9058e-01,  9.9242e-01,  9.9414e-01,
         9.9574e-01,  9.9723e-01,  9.9859e-01,  9.9984e-01,  1.0010e+00,
         1.0020e+00,  1.0028e+00,  1.0036e+00,  1.0042e+00,  1.0047e+00,
         1.0051e+00,  1.0053e+00,  1.0054e+00,  1.0054e+00,  1.0052e+00,
         1.0049e+00,  1.0045e+00,  1.0040e+00,  1.0033e+00,  1.0025e+00,
         1.0015e+00,  1.0004e+00,  9.9919e-01,  9.9781e-01,  9.9630e-01,
         9.9465e-01,  9.9286e-01,  9.9093e-01,  9.8886e-01,  9.8664e-01,
         9.8428e-01,  9.8178e-01,  9.7913e-01,  9.7634e-01,  9.7339e-01,
         9.7031e-01,  9.6708e-01,  9.6370e-01,  9.6018e-01,  9.5650e-01,
         9.5268e-01,  9.4871e-01,  9.4458e-01,  9.4032e-01,  9.3589e-01,
         9.3133e-01,  9.2661e-01,  9.2176e-01,  9.1

In [20]:
sampled_functions

tensor([[ 0.9421,  0.9459,  0.9495,  ..., -0.4643, -0.4742, -0.4840],
        [-0.5627, -0.5719, -0.5810,  ..., -0.5946, -0.5873, -0.5799],
        [-0.3765, -0.3869, -0.3973,  ..., -0.5832, -0.5730, -0.5627],
        ...,
        [-0.3765, -0.3869, -0.3973,  ..., -0.5832, -0.5730, -0.5627],
        [ 0.9907,  0.9879,  0.9849,  ..., -0.9446, -0.9441, -0.9433],
        [-0.4316, -0.4402, -0.4488,  ..., -0.7637, -0.7577, -0.7516]],
       grad_fn=<StackBackward0>)

In [21]:
sampled_functions.shape

torch.Size([10, 200])

In [22]:
# ── Step 8: Interpolate edge functions at X_train and X_test ──
H_train = torch.zeros(N_train, layer_width)
H_test  = torch.zeros(N_test,  layer_width)

for i in range(layer_width):
    seg_x = x_segments[i, :, 0].detach().numpy()   # (T_sample,) — x positions of edge i
    seg_f = sampled_functions[i].detach().numpy()   # (T_sample,) — φi values at those x positions

    x_train_np = X_train[:, 0].detach().numpy()    # (N_train,)
    x_test_np  = X_test[:, 0].detach().numpy()     # (N_test,)

    # For each training point: interpolate φi(x) from the lookup table
    H_train[:, i] = torch.tensor(np.interp(x_train_np, seg_x, seg_f))
    H_test[:, i]  = torch.tensor(np.interp(x_test_np,  seg_x, seg_f))

In [23]:
layer_width

10

In [24]:
x_train_np

array([-3.        , -2.939394  , -2.878788  , -2.8181818 , -2.7575758 ,
       -2.6969697 , -2.6363635 , -2.5757575 , -2.5151515 , -2.4545455 ,
       -2.3939395 , -2.3333333 , -2.2727273 , -2.2121212 , -2.151515  ,
       -2.090909  , -2.030303  , -1.969697  , -1.9090909 , -1.8484848 ,
       -1.7878788 , -1.7272727 , -1.6666666 , -1.6060605 , -1.5454545 ,
       -1.4848485 , -1.4242424 , -1.3636363 , -1.3030303 , -1.2424242 ,
       -1.1818181 , -1.121212  , -1.060606  , -0.99999994, -0.9393939 ,
       -0.8787878 , -0.81818175, -0.7575757 , -0.6969696 , -0.63636357,
       -0.5757575 , -0.51515144, -0.45454538, -0.39393932, -0.33333325,
       -0.2727272 , -0.21212113, -0.15151507, -0.090909  , -0.03030294,
        0.03030294,  0.090909  ,  0.15151507,  0.21212113,  0.2727272 ,
        0.33333325,  0.39393932,  0.45454538,  0.51515144,  0.5757575 ,
        0.63636357,  0.6969696 ,  0.7575757 ,  0.81818175,  0.8787878 ,
        0.9393939 ,  0.99999994,  1.060606  ,  1.121212  ,  1.18

In [25]:
H_train

tensor([[ 0.9421, -0.5627, -0.3765,  0.9410, -0.9453, -0.2587,  0.9781, -0.3765,
          0.9907, -0.4316],
        [ 0.9421, -0.5627, -0.3765,  0.9410, -0.9453, -0.2587,  0.9781, -0.3765,
          0.9907, -0.4316],
        [ 0.9421, -0.5627, -0.3765,  0.9410, -0.9453, -0.2587,  0.9781, -0.3765,
          0.9907, -0.4316],
        [ 0.9421, -0.5627, -0.3765,  0.9410, -0.9453, -0.2587,  0.9781, -0.3765,
          0.9907, -0.4316],
        [ 0.9421, -0.5627, -0.3765,  0.9410, -0.9453, -0.2587,  0.9781, -0.3765,
          0.9907, -0.4316],
        [ 0.9421, -0.5627, -0.3765,  0.9410, -0.9453, -0.2587,  0.9781, -0.3765,
          0.9907, -0.4316],
        [ 0.9421, -0.5627, -0.3765,  0.9410, -0.9453, -0.2587,  0.9781, -0.3765,
          0.9907, -0.4316],
        [ 0.9421, -0.5627, -0.3765,  0.9410, -0.9453, -0.2587,  0.9781, -0.3765,
          0.9907, -0.4316],
        [ 0.9421, -0.6166, -0.5627,  0.9410, -0.9453, -0.2587,  0.9781, -0.5627,
          0.9907, -0.4316],
        [ 0.9421, -

In [26]:
H_train.shape

torch.Size([100, 10])

In [27]:
# ── Step 9: OLS — solve for output layer ──
H_train_b = torch.cat([H_train, torch.ones(N_train, 1)], dim=1)  # (N_train, layer_width+1)
H_test_b  = torch.cat([H_test,  torch.ones(N_test,  1)], dim=1)  # (N_test,  layer_width+1)

result = torch.linalg.lstsq(H_train_b, y_train.unsqueeze(1))
W_out  = result.solution  # (layer_width+1, 1)

# ── Step 10: Predict and evaluate ──
y_pred = H_test_b @ W_out
mse    = ((y_pred.squeeze() - y_test) ** 2).mean()
print(f"\nTest MSE: {mse.item():.6f}")


Test MSE: 0.213277


In [28]:
# Baseline 1: GP posterior mean directly
with torch.no_grad():
    gp_pred = likelihood(model(X_test)).mean
    gp_mse  = ((gp_pred - y_test) ** 2).mean()
    print(f"GP baseline MSE:   {gp_mse.item():.6f}")

# Baseline 2: predicting mean of y_train
# double check
mean_pred = y_train.mean().expand(N_test)
mean_mse  = ((mean_pred - y_test) ** 2).mean()
print(f"Mean baseline MSE: {mean_mse.item():.6f}")

GP baseline MSE:   0.012779
Mean baseline MSE: 0.438664


In [29]:
# Relative L2 error = ||y_pred - y_test||_2 / ||y_test||_2
rel_l2 = torch.norm(y_pred.squeeze() - y_test) / torch.norm(y_test)
print(f"Relative L2 error: {rel_l2.item():.6f}")

# For all baselines too
gp_rel_l2   = torch.norm(gp_pred - y_test) / torch.norm(y_test)
mean_rel_l2 = torch.norm(mean_pred - y_test) / torch.norm(y_test)

print(f"GP baseline relative L2:   {gp_rel_l2.item():.6f}")
print(f"Mean baseline relative L2: {mean_rel_l2.item():.6f}")

Relative L2 error: 0.697307
GP baseline relative L2:   0.170689
Mean baseline relative L2: 1.000041
